In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from functools import partial

In [2]:
def pipeline(*steps):
    def wrapper(inputs):
        for step in steps:
            inputs = step(inputs)
            return inputs
    return wrapper

In [3]:
def step1(df):
    lista_paises = df['Country Name'].drop_duplicates().to_list()
    return lista_paises

In [4]:
def step2(df):
    lista_cols = []
    cols_base = ['Country Name','Country Code','year', 'count_null', 'crisis_pred']
    for col in df.columns:
        if col not in cols_base:
            lista_cols.append(col)
    return lista_cols
    

In [5]:
def contador_nulos(df, lista_paises, lista_cols,valor):
    rows = []
    for pais in lista_paises:
        for col in lista_cols:
            porc_nulos_col = round(df.loc[df['Country Name'] == pais,col].isna().mean(),2)
            if porc_nulos_col > valor:
                rows.append((pais,col,porc_nulos_col))

    
                        
    return pd.DataFrame(rows,columns=['country','indicator','media_null'])

In [6]:
def contador_nulos_wrapper(df,valor=0.70):
    lista_paises = step1(df)
    lista_cols = step2(df)
    df_null = contador_nulos(df, lista_paises, lista_cols,valor)
    paises_null = df_null['country'].drop_duplicates().to_list()
    return df_null,paises_null

In [7]:
df = pd.read_excel('df_top_68.xlsx')

In [ ]:
p = pipeline(partial(contador_nulos_wrapper,valor=0.80))

df_null, paises_null = p(df)

In [9]:
paises_null

['Argentina',
 'Australia',
 'Azerbaijan',
 'Belize',
 'Cameroon',
 'Canada',
 'Chile',
 'Congo, Rep.',
 'Czechia',
 'Dominican Republic',
 'Ecuador',
 'Egypt, Arab Rep.',
 'El Salvador',
 'Gabon',
 'Gambia, The',
 'Ghana',
 'India',
 'Jordan',
 'Kazakhstan',
 'Kenya',
 'Korea, Rep.',
 'Madagascar',
 'Malaysia',
 'Mali',
 'Mauritania',
 'Mongolia',
 'Morocco',
 'Nepal',
 'Niger',
 'Senegal',
 'Sierra Leone',
 'Singapore',
 'Tunisia',
 'Turkiye',
 'United Kingdom',
 'United States',
 'Uruguay']

In [10]:
df_null

,country,indicator,media_null
0,Argentina,"Inflation, consumer prices (annual %)",0.87
1,Australia,External debt stocks (% of GNI),1.00
2,Australia,Short-term debt (% of total external debt),1.00
3,Australia,"Total debt service (% of exports of goods, ser...",1.00
4,Azerbaijan,Bank nonperforming loans to total gross loans (%),0.94
...,...,...,...
82,United States,"Total debt service (% of exports of goods, ser...",1.00
83,Uruguay,Bank nonperforming loans to total gross loans (%),0.84
84,Uruguay,External debt stocks (% of GNI),1.00
85,Uruguay,Short-term debt (% of total external debt),1.00


In [9]:
def cols_nulos(df,lista_cols):
    
    df["count_null"] = df[lista_cols].isna().sum(axis=1)
    df["some_null"]  = (df["count_null"] > 0).astype(int)
    return df

In [10]:
def cols_nulos_wrapper(df):
    lista_cols = step2(df)
    return cols_nulos(df,lista_cols)

In [11]:
p = pipeline(cols_nulos_wrapper)

df = p(df)
df

,Unnamed: 0,Country Name,Country Code,year,Bank nonperforming loans to total gross loans (%),Broad money (% of GDP),Current account balance (% of GDP),Deposit interest rate (%),Domestic credit to private sector (% of GDP),Domestic credit to private sector by banks (% of GDP),...,Net barter terms of trade index (2015 = 100),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Short-term debt (% of total external debt),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,Trade (% of GDP),"Unemployment, total (% of total labor force) (national estimate)",count_null,some_null
0,165,Albania,ALB,1980,NaN,NaN,1.013876,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,47.494093,5.600,18,1
1,166,Albania,ALB,1981,NaN,NaN,2.488694,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,46.100466,5.100,15,1
2,167,Albania,ALB,1982,NaN,NaN,-3.589153,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,44.810562,3.600,15,1
3,168,Albania,ALB,1983,NaN,NaN,-2.035704,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,40.410668,4.400,15,1
4,169,Albania,ALB,1984,NaN,NaN,-1.512918,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,38.115683,5.700,14,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518,13426,Uruguay,URY,2020,2.335771,59.126896,-0.641294,4.794616,27.884637,27.847128,...,107.6,42.013292,0.458651,NaN,NaN,12.809403,47.731873,10.413,3,1
3519,13427,Uruguay,URY,2021,1.326336,58.750620,-2.433511,3.799038,26.712977,26.678879,...,103.9,43.554575,-3.262141,NaN,NaN,8.983196,58.315041,9.328,3,1
3520,13428,Uruguay,URY,2022,1.386676,52.557440,-3.522450,6.235004,26.623834,26.608468,...,97.3,41.171083,5.437763,NaN,NaN,6.549880,61.030795,7.877,3,1
3521,13429,Uruguay,URY,2023,1.650773,51.123443,-3.044701,8.101361,28.577104,28.560787,...,102.8,38.823917,8.202351,NaN,NaN,7.206551,52.910987,8.355,3,1


In [12]:
def relleno_nulos_media_pais(df,lista_paises,lista_cols, how = 'mean'):
    from pandas.api.types import is_numeric_dtype

    if how == 'mean':   
        for pais in lista_paises:
            for col in lista_cols:
                if is_numeric_dtype(df[col]):
                    media_col_x_pais = round(df.loc[(df['Country Name'] == pais),col].mean(),6)
                    if np.isnan(media_col_x_pais):
                        media_col_x_pais = round(df[col].median(),6)
                    filtro = df['Country Name'] == pais
                    df.loc[filtro, col] = df.loc[filtro, col].fillna(media_col_x_pais)
                else:
                    continue
        print(df.isna().mean())
        return df
    elif how == 'median':
        for pais in lista_paises:
            for col in lista_cols:
                if is_numeric_dtype(df[col]):
                    media_col_x_pais = round(df.loc[(df['Country Name'] == pais),col].median(),6)
                    if np.isnan(media_col_x_pais):
                        media_col_x_pais = round(df[col].median(),6)
                    filtro = df['Country Name'] == pais
                    df.loc[filtro, col] = df.loc[filtro, col].fillna(media_col_x_pais)
                else:
                    continue
        print(df.isna().mean())
        return df

In [15]:
def relleno_nulos_wrapper(df,how = 'mean'):
    lista_paises = step1(df)
    lista_cols = step2(df)
    return relleno_nulos_media_pais(df,lista_paises,lista_cols, how)



In [14]:
p = pipeline(partial(relleno_nulos_wrapper,how='mean'))

p(df)

Unnamed: 0                                                                 0.0
Country Name                                                               0.0
Country Code                                                               0.0
year                                                                       0.0
Bank nonperforming loans to total gross loans (%)                          0.0
Broad money (% of GDP)                                                     0.0
Current account balance (% of GDP)                                         0.0
Deposit interest rate (%)                                                  0.0
Domestic credit to private sector (% of GDP)                               0.0
Domestic credit to private sector by banks (% of GDP)                      0.0
Exports of goods and services (% of GDP)                                   0.0
Exports of goods and services (current US$)                                0.0
External debt stocks (% of GNI)                     

,Unnamed: 0,Country Name,Country Code,year,Bank nonperforming loans to total gross loans (%),Broad money (% of GDP),Current account balance (% of GDP),Deposit interest rate (%),Domestic credit to private sector (% of GDP),Domestic credit to private sector by banks (% of GDP),...,"Inflation, consumer prices (annual %)",Lending interest rate (%),Net barter terms of trade index (2015 = 100),"Official exchange rate (LCU per US$, period average)",Real interest rate (%),Short-term debt (% of total external debt),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,Trade (% of GDP),"Unemployment, total (% of total labor force) (national estimate)"
0,165,Albania,ALB,1980,13.043853,71.357524,1.013876,7.954011,36.532914,22.739692,...,14.359572,11.712405,103.337116,110.776603,2.317705,19.698603,10.148965,5.506006,47.494093,5.600
1,166,Albania,ALB,1981,13.043853,71.357524,2.488694,7.954011,36.532914,22.739692,...,14.359572,11.712405,103.337116,110.776603,2.317705,19.698603,10.148965,5.506006,46.100466,5.100
2,167,Albania,ALB,1982,13.043853,71.357524,-3.589153,7.954011,36.532914,22.739692,...,14.359572,11.712405,103.337116,110.776603,2.317705,19.698603,10.148965,5.506006,44.810562,3.600
3,168,Albania,ALB,1983,13.043853,71.357524,-2.035704,7.954011,36.532914,22.739692,...,14.359572,11.712405,103.337116,110.776603,2.317705,19.698603,10.148965,5.506006,40.410668,4.400
4,169,Albania,ALB,1984,13.043853,71.357524,-1.512918,7.954011,36.532914,22.739692,...,14.359572,11.712405,103.337116,110.776603,2.317705,19.698603,10.148965,5.506006,38.115683,5.700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518,13426,Uruguay,URY,2020,2.335771,59.126896,-0.641294,4.794616,27.884637,27.847128,...,9.756406,11.231259,107.600000,42.013292,0.458651,10.171800,15.631115,12.809403,47.731873,10.413
3519,13427,Uruguay,URY,2021,1.326336,58.750620,-2.433511,3.799038,26.712977,26.678879,...,7.747914,7.450648,103.900000,43.554575,-3.262141,10.171800,15.631115,8.983196,58.315041,9.328
3520,13428,Uruguay,URY,2022,1.386676,52.557440,-3.522450,6.235004,26.623834,26.608468,...,9.104380,10.865362,97.300000,41.171083,5.437763,10.171800,15.631115,6.549880,61.030795,7.877
3521,13429,Uruguay,URY,2023,1.650773,51.123443,-3.044701,8.101361,28.577104,28.560787,...,5.869104,11.895433,102.800000,38.823917,8.202351,10.171800,15.631115,7.206551,52.910987,8.355
